In [1]:
import torch 
import numpy as np 
import h5py
import os
from pathlib import Path
import importlib
import IPython.display as ipd

# import src.spatial_attn_lightning as binaural_lightning 
import yaml
from pytorch_lightning import Trainer, seed_everything

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# torch.set_float32_matmul_precision('medium')
# torch.backends.cuda.matmul.allow_tf32 = True
# torch.backends.cudnn.allow_tf32 = True

In [2]:
config_path = "config/ci_attention_models/ci_word_task_v09_4MGB.yaml"
config = yaml.safe_load(open(config_path, 'r'))

config['num_workers'] = 0
config['hparas']['batch_size'] = 2
config['getting_acts'] = True
# config['audio']['rep_kwargs']['rep_on_gpu'] = True
# print(f"Default lr is {config['hparas']['lr']}")


# seed_everything(0)
# importlib.reload(binaural_lightning)
# module = binaural_lightning.BinauralAttentionModule(config)

In [3]:
import src.audio_transforms as at
import src.custom_modules as cm
importlib.reload(at)


<module 'src.audio_transforms' from '/net/vast-storage/scratch/vast/mcdermott/imgriff/projects/torch_2_aud_attn/src/audio_transforms.py'>

In [4]:
ci_model = cm.AttnAudioInputRepresentation(**config['audio'])

[cimodel] incorporated sigmoid_rate_level_function: {'dynamic_range': [15.0], 'dynamic_range_interval': 0.95, 'rate_max': [250.0], 'rate_spont': [0.1], 'threshold': [-5.0]}
[cimodel] 16 electrodes spaced linearly from 8.125 to 23.875 mm from apex
[cimodel] Determining ANF positions based on ERB-spaced CFs
[cimodel] 40 ANFs spaced linearly from 4.0744 to 28.2240 mm from apex
[cimodel] 40 ANFs with CFs ERB-spaced from 125.0000 to 8000.0000 Hz
tensor_current_spread shape: torch.Size([1, 40, 16])
[tf_fir_resample] `kwargs_fir_lowpass_filter`: {'cutoff': 50, 'dur_fir': 0.05, 'window': ['kaiser', 5.0]}
[fir_lowpass_filter] sr_filt = 4410000.0 Hz
[fir_lowpass_filter] numtaps = 220501 samples
[fir_lowpass_filter] dur_fir = 0.05 seconds
[fir_lowpass_filter] cutoff = 50 Hz
[fir_lowpass_filter] window = ('kaiser', 5.0)
[cimodel] converting audio to subbands using half_cosine_filterbank


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1699449183005/work/aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [5]:
# trainer = Trainer(
#     precision="32",
#     limit_val_batches=0.0,
#     num_nodes=1,
#     # benchmark=True,
#     devices=1, # was gpus=1,
#     # detect_anomaly=True,
#     # strategy="ddp_notebook",
#     accelerator="gpu",
# )
# trainer.fit(module)

In [6]:
## Check dataset 

# train_loader = module.train_dataloader()

In [7]:
# ci_model = module.coch_gram

In [8]:
audio = torch.rand(1,2,110250)

In [9]:
# !nvidia-smi

In [10]:
ci_output, _ = ci_model(audio, None)

Input shape: torch.Size([110250])
Subbands shaepe: torch.Size([1, 16, 110250])
[cimodel] Performing subband envelope extraction with ideal lowpass filter
[tf_fir_resample] interpreted `tensor_input.shape` as [batch, freq=16, time=110250]
Envelopes shaepe: torch.Size([1, 16, 25000])
Compressed envelope shaepe: torch.Size([1, 16, 25000])
Pulsetrain shaepe: torch.Size([1, 16, 25000])
ANF stimulation shaepe: torch.Size([1, 40, 25000])
ANF firing response shaepe: torch.Size([1, 1, 40, 25000])
Final nervegram shaepe: torch.Size([1, 1, 40, 25000])
Input shape: torch.Size([110250])
Subbands shaepe: torch.Size([1, 16, 110250])
[cimodel] Performing subband envelope extraction with ideal lowpass filter
[tf_fir_resample] interpreted `tensor_input.shape` as [batch, freq=16, time=110250]
Envelopes shaepe: torch.Size([1, 16, 25000])
Compressed envelope shaepe: torch.Size([1, 16, 25000])
Pulsetrain shaepe: torch.Size([1, 16, 25000])
ANF stimulation shaepe: torch.Size([1, 40, 25000])
ANF firing respons

In [11]:
ci_output.shape

torch.Size([1, 2, 40, 25000])

In [8]:
# batch = next(iter(train_loader))

RuntimeError: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero.

In [10]:
cue_features, cue_mask_ixs, scene_features, labels = batch


In [7]:
ci_model(cue_features[0,0], None)

NameError: name 'cue_features' is not defined

In [13]:
cue_features

torch.Size([72, 2, 110250])

In [ ]:
aud_transforms = module.audio_transforms

In [ ]:
ix = 1080

cue, target, distractor, labels = dataset[ix]

scene, _ = aud_transforms(target, distractor)

print("cue")
ipd.display(ipd.Audio(cue[0], rate=44100, normalize=False))
print("target")
ipd.display(ipd.Audio(target[0], rate=44100, normalize=False))
print("distractor")
ipd.display(ipd.Audio(distractor[0], rate=44100, normalize=False))
print("scene")
ipd.display(ipd.Audio(scene[0], rate=44100, normalize=False))